# Praktikum Modul 3 AI 2026
## Deep Learning

Author: 5027241103 - Ni'mah Fauziyyah Atok

**Fase yang akan dikerjakan:**
1. **EDA** - Exploratory Data Analysis
2. **Training ResNet** - Custom ResNet dari scratch
3. **Training EfficientNet** - Custom EfficientNet dari scratch
4. **Error Analysis** - Analisis kesalahan prediksi
5. **Inference** - Prediksi pada test set & generate submission CSV

---

## Setup & Import Libraries

In [ ]:
import os, time, random, sys
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageStat

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import transforms

from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    precision_recall_fscore_support
)

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device configuration
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"✓ CUDA available")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print(f"✓ MPS (Apple Silicon) available")
else:
    DEVICE = torch.device("cpu")
    print(f"⚠ Using CPU (slower)")

print(f"Device: {DEVICE}")

## Configuration

In [ ]:
# Path configuration
TRAIN_DIR   = Path("train")
TEST_DIR    = Path("test")
SAMPLE_CSV  = Path("sample_submission.csv")
OUTPUT_CSV  = Path("submission.csv")

EDA_OUTPUT      = Path("eda_outputs")
RESNET_OUTPUT   = Path("outputs_resnet")
EFFICIENT_OUTPUT= Path("outputs_efficient")
ERROR_OUTPUT    = Path("error_analysis_outputs")

for p in [EDA_OUTPUT, RESNET_OUTPUT, EFFICIENT_OUTPUT, ERROR_OUTPUT]:
    p.mkdir(exist_ok=True)

# Classes
CLASSES = [
    "AnnualCrop","Forest","HerbaceousVegetation","Highway",
    "Industrial","Pasture","PermanentCrop","Residential","River","SeaLake"
]
CLASS2IDX = {c: i for i, c in enumerate(CLASSES)}
IDX2CLASS = {i: c for c, i in CLASS2IDX.items()}

# Model hyperparameters
IMG_SIZE    = 64
BATCH_SIZE  = 64
EPOCHS      = 40
LR          = 1e-3
WEIGHT_DECAY= 1e-4
VAL_SPLIT   = 0.15

print(f"✓ Configuration loaded")
print(f"  Classes: {len(CLASSES)}")
print(f"  Image size: {IMG_SIZE}x{IMG_SIZE}")

---
# FASE 1: EXPLORATORY DATA ANALYSIS (EDA)

### 1.1 Load & Analyze Dataset

In [ ]:
def count_per_class(root):
    return {c: len(list((root/c).glob("*.jpg"))) for c in CLASSES if (root/c).exists()}

train_counts = count_per_class(TRAIN_DIR)
total_train = sum(train_counts.values())
test_count = len(list(TEST_DIR.glob("*.jpg")))

print("\n" + "="*50)
print("  DATASET STATISTICS")
print("="*50)
print(f"\n{'Kelas':<25} {'Count':>8} {'%':>8}")
print("-"*42)
for c, n in train_counts.items():
    pct = n / total_train * 100
    print(f"{c:<25} {n:>8} {pct:>7.1f}%")
print("-"*42)
print(f"{'Total Train':<25} {total_train:>8}")
print(f"{'Total Test':<25} {test_count:>8}")
print("="*50)

### 1.2 Distribusi Kelas

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle("Distribusi Kelas – Train Set", fontsize=15, fontweight="bold")

colors = plt.cm.Set3(np.linspace(0, 1, len(CLASSES)))
cls_list = list(train_counts.keys())
cnt_list = list(train_counts.values())

# Bar chart
bars = ax1.bar(cls_list, cnt_list, color=colors, edgecolor="black", lw=0.5)
ax1.set_xticklabels(cls_list, rotation=45, ha="right", fontsize=8)
ax1.set_ylabel("Jumlah Gambar")
ax1.set_title("Bar Chart")
ax1.grid(axis="y", alpha=0.3)
for b, v in zip(bars, cnt_list):
    ax1.text(b.get_x() + b.get_width()/2, b.get_height() + 2, str(v), 
             ha="center", fontsize=7)

# Pie chart
ax2.pie(cnt_list, labels=cls_list, autopct="%1.1f%%", colors=colors,
        textprops={"fontsize": 7}, startangle=90)
ax2.set_title("Pie Chart")

plt.tight_layout()
plt.savefig(EDA_OUTPUT/"01_class_distribution.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: 01_class_distribution.png")

### 1.3 Sample Gambar per Kelas

In [ ]:
def load_samples(cls, n=5):
    files = list((TRAIN_DIR/cls).glob("*.jpg"))
    return [Image.open(f).convert("RGB") for f in random.sample(files, min(n, len(files)))]

n_samples = 5
fig, axes = plt.subplots(len(CLASSES), n_samples, figsize=(n_samples*3, len(CLASSES)*2.8))
fig.suptitle("Sampel Gambar per Kelas", fontsize=14, fontweight="bold")

for r, cls in enumerate(CLASSES):
    for c, img in enumerate(load_samples(cls, n_samples)):
        ax = axes[r][c]
        ax.imshow(img)
        ax.axis("off")
        if c == 0:
            ax.set_ylabel(cls, fontsize=9, rotation=0, labelpad=75, va="center")

plt.tight_layout()
plt.savefig(EDA_OUTPUT/"02_sample_images.png", dpi=120, bbox_inches="tight")
plt.show()
print("✓ Saved: 02_sample_images.png")

### 1.4 Image Sizes Analysis

In [ ]:
ws, hs = [], []
for cls in CLASSES:
    files = list((TRAIN_DIR/cls).glob("*.jpg"))
    for f in random.sample(files, min(20, len(files))):
        w, h = Image.open(f).size
        ws.append(w)
        hs.append(h)

uniq = Counter(zip(ws, hs))
print(f"\nImage Sizes:")
print(f"  Width  : min={min(ws)}, max={max(ws)}, mean={np.mean(ws):.1f}")
print(f"  Height : min={min(hs)}, max={max(hs)}, mean={np.mean(hs):.1f}")
print(f"  Unique sizes: {len(uniq)}")
print(f"  Top-3: {uniq.most_common(3)}")

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Distribusi Ukuran Gambar", fontsize=13, fontweight="bold")

a1.hist(ws, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
a1.set(title="Width", xlabel="pixels", ylabel="Frequency")

a2.hist(hs, bins=20, color="salmon", edgecolor="black", alpha=0.8)
a2.set(title="Height", xlabel="pixels", ylabel="Frequency")

plt.tight_layout()
plt.savefig(EDA_OUTPUT/"03_image_sizes.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: 03_image_sizes.png")

### 1.5 Brightness Analysis per Kelas

In [ ]:
def mean_brightness(cls, n=30):
    files = list((TRAIN_DIR/cls).glob("*.jpg"))
    files = random.sample(files, min(n, len(files)))
    brightness_values = [
        ImageStat.Stat(Image.open(f).convert("L")).mean[0] 
        for f in files
    ]
    return float(np.mean(brightness_values))

brightness_per_class = {c: mean_brightness(c) for c in CLASSES}
items = sorted(brightness_per_class.items(), key=lambda x: x[1])
cls_sorted, val_sorted = zip(*items)

colors = plt.cm.RdYlGn(np.linspace(0.2, 0.9, len(CLASSES)))
fig, ax = plt.subplots(figsize=(12, 5))

bars = ax.barh(cls_sorted, val_sorted, color=colors, edgecolor="black", lw=0.5)
ax.axvline(np.mean(val_sorted), color="red", ls="--", 
           label=f"Mean={np.mean(val_sorted):.1f}")
ax.set(xlabel="Mean Brightness (0-255)", title="Brightness per Kelas")
ax.legend()
ax.grid(axis="x", alpha=0.3)

for b, v in zip(bars, val_sorted):
    ax.text(v + 0.5, b.get_y() + b.get_height()/2, f"{v:.1f}", 
            va="center", fontsize=8)

plt.tight_layout()
plt.savefig(EDA_OUTPUT/"04_brightness.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: 04_brightness.png")

### 1.6 RGB Channel Analysis

In [ ]:
def mean_rgb(cls, n=30):
    files = list((TRAIN_DIR/cls).glob("*.jpg"))
    files = random.sample(files, min(n, len(files)))
    arrays = [np.array(Image.open(f).convert("RGB")) for f in files]
    return tuple(np.mean([a[:, :, i].mean() for a in arrays]) for i in range(3))

rgb_per_class = {c: mean_rgb(c) for c in CLASSES}
r_vals = [rgb_per_class[c][0] for c in CLASSES]
g_vals = [rgb_per_class[c][1] for c in CLASSES]
b_vals = [rgb_per_class[c][2] for c in CLASSES]

x = np.arange(len(CLASSES))
w = 0.25

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(x - w, r_vals, w, label="Red", color="tomato", alpha=0.85, edgecolor="k")
ax.bar(x, g_vals, w, label="Green", color="mediumseagreen", alpha=0.85, edgecolor="k")
ax.bar(x + w, b_vals, w, label="Blue", color="cornflowerblue", alpha=0.85, edgecolor="k")

ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=8)
ax.set(ylabel="Mean Pixel Value", title="Mean RGB per Kelas")
ax.legend()
ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(EDA_OUTPUT/"05_rgb_per_class.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: 05_rgb_per_class.png")

### 1.7 Similar Classes Comparison

In [ ]:
# Pairs of visually similar classes
similar_pairs = [
    ("AnnualCrop", "PermanentCrop"),
    ("Forest", "HerbaceousVegetation"),
    ("River", "SeaLake"),
    ("Highway", "Residential"),
    ("Pasture", "HerbaceousVegetation")
]

fig, axes = plt.subplots(len(similar_pairs), 6, figsize=(18, len(similar_pairs)*3))
fig.suptitle("Perbandingan Kelas Mirip Secara Visual", fontsize=13, fontweight="bold")

for row, (c1, c2) in enumerate(similar_pairs):
    for col, img in enumerate(load_samples(c1, 3)):
        axes[row][col].imshow(img)
        axes[row][col].axis("off")
        if col == 0:
            axes[row][col].set_title(c1, fontsize=8, color="blue", fontweight="bold")
    
    for col, img in enumerate(load_samples(c2, 3)):
        axes[row][col + 3].imshow(img)
        axes[row][col + 3].axis("off")
        if col == 0:
            axes[row][col + 3].set_title(c2, fontsize=8, color="red", fontweight="bold")

plt.tight_layout()
plt.savefig(EDA_OUTPUT/"06_similar_classes.png", dpi=120, bbox_inches="tight")
plt.show()
print("✓ Saved: 06_similar_classes.png")
print("\n" + "="*50)
print("  EDA COMPLETED ✓")
print("="*50)

---
# FASE 2: DATA PREPARATION & DATALOADERS

### Dataset Class & Transforms

In [ ]:
class SatelliteDataset(Dataset):
    """Base dataset class for satellite images."""
    def __init__(self, root, transform=None):
        self.samples = []
        self.transform = transform
        for cls in CLASSES:
            cls_dir = root / cls
            if not cls_dir.exists():
                continue
            for f in cls_dir.glob("*.jpg"):
                self.samples.append((str(f), CLASS2IDX[cls]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, label

# Training transforms (with augmentation)
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.1),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.75, 1.0)),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Validation transforms (no augmentation)
val_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

print("✓ Dataset class & transforms prepared")

### Build DataLoaders

In [ ]:
class SubsetWithTransform(Dataset):
    """Wrapper for subset with transform."""
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, i):
        raw = self.subset.dataset.samples[self.subset.indices[i]]
        img = Image.open(raw[0]).convert("RGB")
        return self.transform(img), raw[1]

def build_dataloaders():
    """Create train and validation dataloaders."""
    full_ds = SatelliteDataset(TRAIN_DIR, transform=None)
    n_val = int(len(full_ds) * VAL_SPLIT)
    n_train = len(full_ds) - n_val
    
    gen = torch.Generator().manual_seed(SEED)
    tr_subset, vl_subset = random_split(full_ds, [n_train, n_val], generator=gen)

    tr_ds = SubsetWithTransform(tr_subset, train_transforms)
    vl_ds = SubsetWithTransform(vl_subset, val_transforms)

    pin = DEVICE.type == "cuda"
    tr_loader = DataLoader(tr_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=2, pin_memory=pin)
    vl_loader = DataLoader(vl_ds, batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=pin)
    
    return tr_loader, vl_loader, len(tr_ds), len(vl_ds)

tr_loader, vl_loader, n_train, n_val = build_dataloaders()
print(f"✓ DataLoaders created")
print(f"  Train: {n_train}  |  Val: {n_val}")

---
# FASE 3: TRAIN CUSTOM ResNet

### Model Architecture: ResNet

In [ ]:
class ResidualBlock(nn.Module):
    """Basic Residual Block with skip connection."""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride=stride, 
                               padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.downsample = None
        if stride != 1 or in_channels != out_channels:
            self.downsample = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.downsample:
            identity = self.downsample(x)
        return self.relu(out + identity)

class CustomResNet(nn.Module):
    """Custom ResNet built from scratch (no pretrained weights)."""
    def __init__(self, num_classes=10):
        super().__init__()
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        # Residual stages
        self.layer1 = nn.Sequential(
            ResidualBlock(64, 64),
            ResidualBlock(64, 64)
        )
        self.layer2 = nn.Sequential(
            ResidualBlock(64, 128, stride=2),
            ResidualBlock(128, 128)
        )
        self.layer3 = nn.Sequential(
            ResidualBlock(128, 256, stride=2),
            ResidualBlock(256, 256)
        )
        self.layer4 = nn.Sequential(
            ResidualBlock(256, 512, stride=2),
            ResidualBlock(512, 512)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.drop = nn.Dropout(0.4)
        self.fc = nn.Linear(512, num_classes)

        # Weight initialization
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.pool(x).flatten(1)
        return self.fc(self.drop(x))

# Initialize model
model_resnet = CustomResNet(num_classes=len(CLASSES)).to(DEVICE)
n_params_resnet = sum(p.numel() for p in model_resnet.parameters() if p.requires_grad)
print(f"✓ ResNet model created")
print(f"  Parameters: {n_params_resnet:,}")

### Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, scaler):
    """Train for one epoch."""
    model.train()
    total_loss, correct, total = 0., 0, 0
    
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        
        use_amp = DEVICE.type == "cuda"
        with torch.amp.autocast(device_type="cuda", enabled=use_amp):
            outputs = model(imgs)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        total_loss += loss.item() * imgs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += imgs.size(0)
    
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    """Evaluate for one epoch."""
    model.eval()
    total_loss, correct, total = 0., 0, 0
    all_preds, all_labels = [], []
    
    for imgs, labels in loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        
        total_loss += loss.item() * imgs.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += imgs.size(0)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    
    f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)
    return total_loss / total, correct / total, f1, all_preds, all_labels

print("✓ Training functions prepared")

### Train ResNet

In [ ]:
print("="*60)
print("  TRAINING CUSTOM ResNet")
print(f"  Device: {DEVICE}  |  Epochs: {EPOCHS}")
print("="*60)

criterion_resnet = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_resnet = optim.AdamW(model_resnet.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_resnet = optim.lr_scheduler.CosineAnnealingLR(optimizer_resnet, T_max=EPOCHS, eta_min=1e-5)
scaler_resnet = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

history_resnet = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc", "val_f1"]}
best_f1_resnet = 0.
best_path_resnet = RESNET_OUTPUT / "resnet_best.pth"

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    tr_loss, tr_acc = train_epoch(model_resnet, tr_loader, criterion_resnet, 
                                  optimizer_resnet, scaler_resnet)
    vl_loss, vl_acc, vl_f1, preds, labels = eval_epoch(model_resnet, vl_loader, 
                                                        criterion_resnet)
    scheduler_resnet.step()
    
    for k, v in zip(["train_loss", "val_loss", "train_acc", "val_acc", "val_f1"],
                    [tr_loss, vl_loss, tr_acc, vl_acc, vl_f1]):
        history_resnet[k].append(v)
    
    mark = "✓" if vl_f1 > best_f1_resnet else " "
    if vl_f1 > best_f1_resnet:
        best_f1_resnet = vl_f1
        torch.save(model_resnet.state_dict(), best_path_resnet)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Ep {epoch:>2}/{EPOCHS} | L={tr_loss:.4f}/{vl_loss:.4f} | "
              f"Acc={tr_acc:.3f}/{vl_acc:.3f} | F1={vl_f1:.4f} {mark} | {time.time()-t0:.1f}s")

print(f"\n[DONE] Best Val Macro F1: {best_f1_resnet:.4f}")
print(f"       Model saved: {best_path_resnet}")

# Reload best & final eval
model_resnet.load_state_dict(torch.load(best_path_resnet, map_location=DEVICE))
_, _, best_f1_check, final_preds_resnet, final_labels_resnet = eval_epoch(
    model_resnet, vl_loader, criterion_resnet
)

print("\n" + "="*60)
print("RESNET - Classification Report")
print("="*60)
print(classification_report(final_labels_resnet, final_preds_resnet, 
                          target_names=CLASSES, digits=4))

### ResNet Training Plots

In [ ]:
# Training history plots
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Custom ResNet – Training History", fontsize=13, fontweight="bold")

keys = [("train_loss", "val_loss", "Loss"), 
        ("train_acc", "val_acc", "Accuracy"), 
        ("val_f1", None, "Val Macro F1")]

for ax, (k1, k2, title) in zip(axes, keys):
    ax.plot(history_resnet[k1], label="Train", linewidth=2)
    if k2:
        ax.plot(history_resnet[k2], label="Val", linewidth=2)
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(RESNET_OUTPUT / "training_history.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: training_history.png")

In [ ]:
# Confusion matrix
cm_resnet = confusion_matrix(final_labels_resnet, final_preds_resnet)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_resnet, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax, 
            annot_kws={"size": 8})
ax.set_xlabel("Predicted", fontsize=10)
ax.set_ylabel("True", fontsize=10)
ax.set_title("Confusion Matrix – Custom ResNet", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(RESNET_OUTPUT / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: confusion_matrix.png")

---
# FASE 4: TRAIN CUSTOM EfficientNet

### Model Architecture: EfficientNet

In [ ]:
class SqueezeExciteBlock(nn.Module):
    """SE block for channel-wise attention."""
    def __init__(self, channels, reduction=4):
        super().__init__()
        self.se = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction, bias=False),
            nn.SiLU(),
            nn.Linear(channels // reduction, channels, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        return x * self.se(x).view(x.size(0), -1, 1, 1)

class MBConvBlock(nn.Module):
    """Mobile Inverted Residual (MBConv) block with SE."""
    def __init__(self, in_channels, out_channels, expand_ratio=4, stride=1):
        super().__init__()
        mid_channels = in_channels * expand_ratio
        self.use_residual = (in_channels == out_channels and stride == 1)
        
        # Expansion phase
        self.expand = nn.Sequential(
            nn.Conv2d(in_channels, mid_channels, 1, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.SiLU()
        ) if expand_ratio != 1 else nn.Identity()
        
        # Depthwise phase
        self.dw = nn.Sequential(
            nn.Conv2d(mid_channels, mid_channels, 3, stride=stride, padding=1,
                      groups=mid_channels, bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.SiLU()
        )
        
        # SE block
        self.se = SqueezeExciteBlock(mid_channels, reduction=4)
        
        # Pointwise phase
        self.pw = nn.Sequential(
            nn.Conv2d(mid_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels)
        )

    def forward(self, x):
        out = self.expand(x)
        out = self.dw(out)
        out = self.se(out)
        out = self.pw(out)
        return out + x if self.use_residual else out

class CustomEfficientNet(nn.Module):
    """Custom EfficientNet-inspired architecture from scratch."""
    def __init__(self, num_classes=10):
        super().__init__()
        
        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU()
        )
        
        # MBConv blocks: (in_ch, out_ch, expand, stride)
        config = [
            (32,  16,  1, 1),
            (16,  24,  6, 1),
            (24,  40,  6, 2),
            (40,  80,  6, 2),
            (80,  112, 6, 1),
            (112, 192, 6, 2),
            (192, 320, 6, 1),
        ]
        
        layers = []
        for in_ch, out_ch, exp, st in config:
            layers.append(MBConvBlock(in_ch, out_ch, expand_ratio=exp, stride=st))
        self.blocks = nn.Sequential(*layers)
        
        # Head
        self.head = nn.Sequential(
            nn.Conv2d(320, 1280, 1, bias=False),
            nn.BatchNorm2d(1280),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(1280, num_classes)
        )
        
        self._init_weights()

    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        return self.head(x)

# Initialize model
model_efficient = CustomEfficientNet(num_classes=len(CLASSES)).to(DEVICE)
n_params_efficient = sum(p.numel() for p in model_efficient.parameters() if p.requires_grad)
print(f"✓ EfficientNet model created")
print(f"  Parameters: {n_params_efficient:,}")

### Train EfficientNet

In [ ]:
print("="*60)
print("  TRAINING CUSTOM EfficientNet")
print(f"  Device: {DEVICE}  |  Epochs: {EPOCHS}")
print("="*60)

criterion_efficient = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer_efficient = optim.AdamW(model_efficient.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler_efficient = optim.lr_scheduler.OneCycleLR(
    optimizer_efficient, max_lr=LR*5, epochs=EPOCHS,
    steps_per_epoch=len(tr_loader), pct_start=0.1
)
scaler_efficient = torch.amp.GradScaler(enabled=DEVICE.type == "cuda")

history_efficient = {k: [] for k in ["train_loss", "val_loss", "train_acc", "val_acc", "val_f1"]}
best_f1_efficient = 0.
best_path_efficient = EFFICIENT_OUTPUT / "efficient_best.pth"

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    
    tr_loss, tr_acc = train_epoch(model_efficient, tr_loader, criterion_efficient,
                                  optimizer_efficient, scaler_efficient)
    vl_loss, vl_acc, vl_f1, preds, labels = eval_epoch(model_efficient, vl_loader,
                                                        criterion_efficient)
    scheduler_efficient.step()
    
    for k, v in zip(["train_loss", "val_loss", "train_acc", "val_acc", "val_f1"],
                    [tr_loss, vl_loss, tr_acc, vl_acc, vl_f1]):
        history_efficient[k].append(v)
    
    mark = "✓" if vl_f1 > best_f1_efficient else " "
    if vl_f1 > best_f1_efficient:
        best_f1_efficient = vl_f1
        torch.save(model_efficient.state_dict(), best_path_efficient)
    
    if epoch % 5 == 0 or epoch == 1:
        print(f"Ep {epoch:>2}/{EPOCHS} | L={tr_loss:.4f}/{vl_loss:.4f} | "
              f"Acc={tr_acc:.3f}/{vl_acc:.3f} | F1={vl_f1:.4f} {mark} | {time.time()-t0:.1f}s")

print(f"\n[DONE] Best Val Macro F1: {best_f1_efficient:.4f}")
print(f"       Model saved: {best_path_efficient}")

# Reload best & final eval
model_efficient.load_state_dict(torch.load(best_path_efficient, map_location=DEVICE))
_, _, best_f1_check, final_preds_efficient, final_labels_efficient = eval_epoch(
    model_efficient, vl_loader, criterion_efficient
)

print("\n" + "="*60)
print("EFFICIENTNET - Classification Report")
print("="*60)
print(classification_report(final_labels_efficient, final_preds_efficient,
                          target_names=CLASSES, digits=4))

### EfficientNet Training Plots

In [ ]:
# Training history plots
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("Custom EfficientNet – Training History", fontsize=13, fontweight="bold")

keys = [("train_loss", "val_loss", "Loss"),
        ("train_acc", "val_acc", "Accuracy"),
        ("val_f1", None, "Val Macro F1")]

for ax, (k1, k2, title) in zip(axes, keys):
    ax.plot(history_efficient[k1], label="Train", linewidth=2)
    if k2:
        ax.plot(history_efficient[k2], label="Val", linewidth=2)
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(EFFICIENT_OUTPUT / "training_history.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: training_history.png")

In [ ]:
# Confusion matrix
cm_efficient = confusion_matrix(final_labels_efficient, final_preds_efficient)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm_efficient, annot=True, fmt="d", cmap="Greens",
            xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
            annot_kws={"size": 8})
ax.set_xlabel("Predicted", fontsize=10)
ax.set_ylabel("True", fontsize=10)
ax.set_title("Confusion Matrix – Custom EfficientNet", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(EFFICIENT_OUTPUT / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: confusion_matrix.png")

---
# FASE 5: ERROR ANALYSIS

### Validation Set Predictions

In [ ]:
# Get all validation data with paths for error analysis
class ValDatasetWithPath(Dataset):
    def __init__(self, samples, transform):
        self.samples = samples  # list of (path, label)
        self.transform = transform
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert("RGB")
        return self.transform(img), label, path

def build_val_loader_with_paths():
    """Build validation loader with image paths."""
    full_ds = SatelliteDataset(TRAIN_DIR, transform=None)
    n_val = int(len(full_ds) * VAL_SPLIT)
    n_train = len(full_ds) - n_val
    
    gen = torch.Generator().manual_seed(SEED)
    _, vl_subset = random_split(full_ds, [n_train, n_val], generator=gen)
    
    val_samples = [full_ds.samples[i] for i in vl_subset.indices]
    val_ds = ValDatasetWithPath(val_samples, val_transforms)
    
    pin = DEVICE.type == "cuda"
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                           num_workers=2, pin_memory=pin)
    return val_loader

val_loader_with_paths = build_val_loader_with_paths()

@torch.no_grad()
def get_predictions_with_paths(model, loader):
    """Get predictions, labels, paths and probabilities."""
    all_preds, all_labels, all_paths, all_probs = [], [], [], []
    
    for imgs, labels, paths in loader:
        imgs = imgs.to(DEVICE)
        logits = model(imgs)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(1).cpu().numpy()
        
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_paths.extend(paths)
        all_probs.extend(probs)
    
    return (np.array(all_labels), np.array(all_preds), all_paths, np.array(all_probs))

print("✓ Getting predictions from both models...")
labels_val, preds_resnet_val, paths_val, probs_resnet_val = get_predictions_with_paths(
    model_resnet, val_loader_with_paths
)
_, preds_efficient_val, _, probs_efficient_val = get_predictions_with_paths(
    model_efficient, val_loader_with_paths
)

print(f"  Validation set: {len(labels_val)} samples")

### Comparison: Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(22, 8))
fig.suptitle("Confusion Matrix Comparison", fontsize=14, fontweight="bold")

for ax, preds, title, cmap in zip(
    axes,
    [preds_resnet_val, preds_efficient_val],
    ["Custom ResNet", "Custom EfficientNet"],
    ["Blues", "Greens"]
):
    cm = confusion_matrix(labels_val, preds)
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap,
                xticklabels=CLASSES, yticklabels=CLASSES, ax=ax,
                annot_kws={"size": 6})
    ax.set_xlabel("Predicted", fontsize=10)
    ax.set_ylabel("True", fontsize=10)
    ax.set_title(f"Confusion Matrix – {title}", fontsize=11, fontweight="bold")
    ax.tick_params(axis="x", rotation=45, labelsize=7)
    ax.tick_params(axis="y", rotation=0, labelsize=7)

plt.tight_layout()
plt.savefig(ERROR_OUTPUT / "01_confusion_compare.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: 01_confusion_compare.png")

### Per-Class F1 Score Comparison

In [ ]:
_, _, f1_resnet, _ = precision_recall_fscore_support(
    labels_val, preds_resnet_val, average=None, labels=list(range(10)), zero_division=0
)
_, _, f1_efficient, _ = precision_recall_fscore_support(
    labels_val, preds_efficient_val, average=None, labels=list(range(10)), zero_division=0
)

x = np.arange(len(CLASSES))
w = 0.35

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar(x - w/2, f1_resnet, w, label="ResNet", color="steelblue", alpha=0.85, edgecolor="k")
ax.bar(x + w/2, f1_efficient, w, label="EfficientNet", color="mediumseagreen", alpha=0.85, edgecolor="k")

ax.set_xticks(x)
ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=9)
ax.set_ylabel("F1 Score", fontsize=10)
ax.set_title("Per-Class F1 Score Comparison", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(ERROR_OUTPUT / "02_perclass_f1.png", dpi=150, bbox_inches="tight")
plt.show()
print("✓ Saved: 02_perclass_f1.png")

### Misclassified Samples

def plot_misclassified(labels, preds, paths, probs, model_name, n=16):
    """Plot misclassified samples."""
    wrong_idx = np.where(labels != preds)[0]
    if len(wrong_idx) == 0:
        print(f"  No misclassifications for {model_name}!")
        return
    
    sampled = np.random.choice(wrong_idx, min(n, len(wrong_idx)), replace=False)
    
    cols = 4
    rows = (len(sampled) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols*3.5, rows*3.5))
    fig.suptitle(f"Misclassified Samples – {model_name}", fontsize=13, fontweight="bold")
    
    axes_flat = axes.flatten() if hasattr(axes, "flatten") else [axes]
    
    for ax_idx, i in enumerate(sampled):
        ax = axes_flat[ax_idx]
        img = Image.open(paths[i]).convert("RGB")
        ax.imshow(img)
        
        true_cls = IDX2CLASS[int(labels[i])]
        pred_cls = IDX2CLASS[int(preds[i])]
        conf = probs[i][int(preds[i])] * 100
        
        ax.set_title(f"True: {true_cls}\nPred: {pred_cls} ({conf:.1f}%)",
                    fontsize=7, color="red")
        ax.axis("off")
    
    for ax in axes_flat[len(sampled):]:
        ax.axis("off")
    
    plt.tight_layout()
    fname = f"03_misclassified_{model_name.lower().replace(' ', '_')}.png"
    plt.savefig(ERROR_OUTPUT / fname, dpi=130, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {fname}")

plot_misclassified(labels_val, preds_resnet_val, paths_val, probs_resnet_val, "resnet")
plot_misclassified(labels_val, preds_efficient_val, paths_val, probs_efficient_val, "efficientnet")

### Most Confused Pairs

In [ ]:
def plot_confused_pairs(labels, preds, model_name):
    """Plot top-10 most confused class pairs."""
    cm = confusion_matrix(labels, preds)
    np.fill_diagonal(cm, 0)
    
    pairs = [(cm[i, j], CLASSES[i], CLASSES[j])
             for i in range(len(CLASSES)) for j in range(len(CLASSES))
             if i != j and cm[i, j] > 0]
    pairs.sort(reverse=True)
    top10 = pairs[:10]
    
    labs = [f"{t}→{p}" for _, t, p in top10]
    vals = [v for v, _, _ in top10]
    
    fig, ax = plt.subplots(figsize=(12, 5))
    bars = ax.barh(labs[::-1], vals[::-1], color="salmon", edgecolor="k")
    ax.set_xlabel("Count", fontsize=10)
    ax.set_title(f"Top-10 Most Confused Pairs – {model_name}", fontsize=12, fontweight="bold")
    ax.grid(axis="x", alpha=0.3)
    
    for b, v in zip(bars, vals[::-1]):
        ax.text(v + 0.2, b.get_y() + b.get_height()/2, str(v), va="center", fontsize=9)
    
    plt.tight_layout()
    fname = f"04_confused_pairs_{model_name.lower()}.png"
    plt.savefig(ERROR_OUTPUT / fname, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"✓ Saved: {fname}")

plot_confused_pairs(labels_val, preds_resnet_val, "resnet")
plot_confused_pairs(labels_val, preds_efficient_val, "efficientnet")

### Error Analysis Summary

In [ ]:
f1_macro_resnet = f1_score(labels_val, preds_resnet_val, average="macro", zero_division=0)
f1_macro_efficient = f1_score(labels_val, preds_efficient_val, average="macro", zero_division=0)

acc_resnet = (labels_val == preds_resnet_val).mean()
acc_efficient = (labels_val == preds_efficient_val).mean()

print("\n" + "="*60)
print("  ERROR ANALYSIS SUMMARY")
print("="*60)
print(f"\n{'Model':<25} {'Accuracy':>12} {'Macro F1':>12}")
print("-"*50)
print(f"{'Custom ResNet':<25} {acc_resnet:>12.4f} {f1_macro_resnet:>12.4f}")
print(f"{'Custom EfficientNet':<25} {acc_efficient:>12.4f} {f1_macro_efficient:>12.4f}")
print("-"*50)

best_model = "ResNet" if f1_macro_resnet >= f1_macro_efficient else "EfficientNet"
best_f1 = max(f1_macro_resnet, f1_macro_efficient)
print(f"\n>>> Best Model: {best_model} (F1={best_f1:.4f})")
print("="*60)

print("\n--- ResNet per-class report ---")
print(classification_report(labels_val, preds_resnet_val, target_names=CLASSES, digits=4))

print("\n--- EfficientNet per-class report ---")
print(classification_report(labels_val, preds_efficient_val, target_names=CLASSES, digits=4))

---
# FASE 6: INFERENCE & SUBMISSION

### Inference on Test Set

print("="*60)
print("  INFERENCE ON TEST SET")
print("="*60)

# Load test files
test_files = sorted(TEST_DIR.glob("*.jpg"))
if not test_files:
    print(f"[ERROR] No images in {TEST_DIR}")
else:
    print(f"\nTest images: {len(test_files)}")
    
    # Load sample submission
    sample_df = pd.read_csv(SAMPLE_CSV)
    print(f"Sample rows: {len(sample_df)}")
    
    # Predict with both models using TTA
    print("\n[Predicting with TTA...]")
    print("  - ResNet (4 augmentations)...")
    probs_resnet = predict_with_tta(model_resnet, test_files)
    
    print("  - EfficientNet (4 augmentations)...")
    probs_efficient = predict_with_tta(model_efficient, test_files)
    
    # Ensemble: average predictions
    print("  - Ensemble (averaging)...")
    probs_ensemble = (probs_resnet + probs_efficient) / 2
    preds_final = probs_ensemble.argmax(axis=1)
    
    # Map to class names
    img_names = [f.name for f in test_files]
    pred_map = {name: IDX2CLASS[int(p)] for name, p in zip(img_names, preds_final)}
    
    # Align with sample submission
    submission = sample_df.copy()
    submission["label"] = submission["image_id"].map(pred_map)
    
    # Handle any missing mappings
    missing = submission["label"].isna().sum()
    if missing > 0:
        print(f"[WARN] {missing} image_id not matched")
        submission["label"] = submission["label"].fillna(
            pd.Series(np.random.choice(CLASSES, size=len(submission)))
        )
    
    # Save submission
    submission.to_csv(OUTPUT_CSV, index=False)
    
    print(f"\n{'='*60}")
    print(f"  SUBMISSION SAVED")
    print(f"{'='*60}")
    print(f"File: {OUTPUT_CSV}")
    print(f"Total rows: {len(submission)}")
    print(f"\nClass distribution:")
    
    dist = submission["label"].value_counts()
    for cls in CLASSES:
        cnt = dist.get(cls, 0)
        bar = "█" * int(cnt / len(submission) * 30)
        print(f"  {cls:<25} {cnt:>5}  {bar}")
    
    print(f"\n--- Preview (first 5 rows) ---")
    print(submission.head().to_string(index=False))
    print(f"{'='*60}")

---
## Summary

**Notebook ini berisi seluruh pipeline klasifikasi gambar satelit:**

1. **EDA** - Analisis distribusi kelas, ukuran gambar, brightness, RGB values
2. **Training ResNet** - Custom ResNet dari scratch dengan 40 epochs
3. **Training EfficientNet** - Custom EfficientNet dengan MBConv blocks
4. **Error Analysis** - Membandingkan performa kedua model
5. **Inference** - Prediksi dengan TTA dan ensemble
6. **Submission** - Generate submission CSV untuk Kaggle

**Output:**
- `eda_outputs/` - Visualisasi EDA
- `outputs_resnet/` - Model ResNet dan training plots
- `outputs_efficient/` - Model EfficientNet dan training plots
- `error_analysis_outputs/` - Analisis perbandingan model
- `submission.csv` - File submission untuk Kaggle

**Catatan:** Untuk menjalankan notebook ini, pastikan sudah memiliki:
- Folder `train/` dengan gambar training
- Folder `test/` dengan gambar testing
- File `sample_submission.csv`